# 04 · Domain Dominance

Who owns HN's front page? We track which domains (news sites, platforms, personal blogs) dominated each era — and how the ecosystem has shifted from individual blogs to corporate publishing platforms and back to newsletters.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from src.loader import db
from src.nlp import extract_domain
from src.viz import set_style, save

set_style()
con = db()

In [ ]:
stories = con.execute("""
    SELECT url, score, YEAR(posted_at) AS year
    FROM stories
    WHERE url IS NOT NULL
      AND score >= 10
      AND YEAR(posted_at) BETWEEN 2008 AND 2024
""").df()

stories['domain'] = stories['url'].apply(extract_domain)
print(f'{len(stories):,} stories with URLs')

## Top domains overall

In [ ]:
top_domains = (
    stories.groupby('domain')
    .agg(n_stories=('score', 'count'), avg_score=('score', 'mean'))
    .sort_values('n_stories', ascending=False)
    .head(30)
)
display(top_domains)

## Domain share over time (top 10 platforms)

In [ ]:
TRACKED = [
    'github.com', 'medium.com', 'nytimes.com', 'techcrunch.com',
    'arstechnica.com', 'substack.com', 'wired.com', 'bloomberg.com',
    'youtube.com', 'reddit.com'
]

total_per_year = stories.groupby('year').size().rename('total')
tracked = stories[stories['domain'].isin(TRACKED)]
pivot = (
    tracked.groupby(['year', 'domain']).size()
    .unstack(fill_value=0)
    .div(total_per_year, axis=0) * 100
)

fig, ax = plt.subplots(figsize=(14, 6))
pivot.plot(ax=ax, linewidth=2, marker='o', markersize=4)
ax.set_title('Domain share of HN stories scoring ≥ 10 (% of year total)', fontsize=14, fontweight='bold')
ax.set_ylabel('Share (%)')
ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.tight_layout()
save(fig, '../data/fig_domain_share.png')
plt.show()

## Top 10 domains per year (ranked table)

In [ ]:
for year in [2010, 2015, 2018, 2021, 2024]:
    top = (
        stories[stories['year'] == year]
        .groupby('domain').size()
        .sort_values(ascending=False)
        .head(10)
    )
    print(f'\n=== {year} ===')
    print(top.to_string())